In [29]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors

# ==============================
# PARÂMETROS GERAIS
# ==============================

L = 100
n_lagartos = L**2
estrategias = ['O', 'Y', 'B']

a = 2
b = 1/a

matriz_payoff = np.array([[1, b, a],
                          [a, 1, b],
                          [b, a, 1]])

index_map = {'O': 0, 'Y': 1, 'B': 2}

n_geracoes = 100
n_pop = 1
A = 0.0
wO = 0
wB = 0
wY = 0
raio = 1
sobreposicao = True

#output_dir = f"C:/Unicamp/mestrado/simulacoes/RPS-python/RPS-POO/outputs/Montecarlo_sobreposicao_geracao/vizinhanca_homogenea/"
if sobreposicao == True:
    output_dir = f"C:/Unicamp/mestrado/simulacoes/RPS-python/RPS-POO/outputs/null-model/" + f"moore_{raio}/com_sobreposicao/"
else:
    output_dir = f"C:/Unicamp/mestrado/simulacoes/RPS-python/RPS-POO/outputs/null-model/" + f"moore_{raio}/sem_sobreposicao/"

os.makedirs(output_dir, exist_ok=True)

# ==============================
# CLASSE LAGARTO
# ==============================

class Lagarto:
    def __init__(self, i, j, raio, estrategia, fitness,
                 coord_vizinhos, coord_vizinhanca_extendida,
                 estrategia_vizinhanca_extendida, n_vizinhos):
        self.i = i
        self.j = j
        self.raio = raio
        self.estrategia = estrategia
        self.fitness = 0
        self.coord_vizinhos = []
        self.coord_vizinhanca_extendida = []
        self.estrategia_vizinhanca_extendida = []
        self.n_vizinhos = 0

    def calcular_coord_vizinhos(self, L):
        r = self.raio

        lista_vizinhos = []
        for dx in range(-r, r+1):
            for dy in range(-r, r+1):
                #if abs(dx) + abs(dy) <= r and not (dx == 0 and dy == 0): #Von neuman
                if not (dx == 0 and dy == 0):
                    ni = (self.i + dx) % L
                    nj = (self.j + dy) % L
                    lista_vizinhos.append((ni, nj))
        self.coord_vizinhos = lista_vizinhos

    def calcular_n_vizinhos(self):
      self.n_vizinhos = len(self.coord_vizinhos) + len(self.coord_vizinhanca_extendida)

    def mortalidade(self, A, w):
        return 1 / (1 + A * np.exp(w * self.fitness))

    def calcular_fitness(self, mapa):
        fitness_total = 0
        todos_vizinhos = set(self.coord_vizinhos + self.coord_vizinhanca_extendida)
        for ni, nj in todos_vizinhos:
            vizinho = mapa.get((ni, nj))
            if vizinho is not None:
                fitness_total += matriz_payoff[index_map[self.estrategia], index_map[vizinho.estrategia]]
        self.fitness = fitness_total
    
    def ajustar_vizinhos_reciprocos(self, mapa): 
        self.coord_vizinhanca_extendida = []
        self.estrategia_vizinhanca_extendida = []
        
        for (ni, nj) in self.coord_vizinhos: 
                vizinho = mapa[(ni, nj)]
                if (self.i, self.j) not in vizinho.coord_vizinhos:
                    vizinho.estrategia_vizinhanca_extendida.append(str(self.estrategia))
                    vizinho.coord_vizinhanca_extendida.append((self.i, self.j))
                    print(f"Ajustando vizinho recíproco: Lagarto ({self.i}, {self.j}) adiciona ({ni}, {nj}) como vizinho estendido")

# ==============================
# FUNÇÕES AUXILIARES
# ==============================

def criar_lagartos():
    lista = []
    for i in range(L):
        for j in range(L):
            estrategia = np.random.choice(estrategias)
            if estrategia == 'O':
                raio = 1
            elif estrategia == 'Y':
                raio = 1
            else:
                raio = 1
            lista.append(Lagarto(i, j, raio, estrategia, 0, [], [], [], 0))
    return lista

def calcular_freq(matriz):
    return np.array([np.sum(matriz == s) / (L**2) for s in estrategias])

def media_vizinhos(lista_lagartos):
    return np.mean([l.n_vizinhos for l in lista_lagartos])

def media_vizinhos_por_estrategia(lista_lagartos):
    medias = []
    for e in estrategias:
        raios = [l.n_vizinhos for l in lista_lagartos if l.estrategia == e]
        medias.append(np.mean(raios) if len(raios) > 0 else np.nan)
    return medias

In [30]:
# ==============================
# SIMULAÇÃO
# ==============================

def simulacao(A, wO, wY, wB, seed=None):

    resultados = []

    for pop in range(n_pop):

        if seed is not None:
            np.random.seed(seed + pop)

        lista_lagartos = criar_lagartos()
        mapa = {(l.i, l.j): l for l in lista_lagartos}

        matriz_posicao = np.empty((L, L), dtype=object)
        for l in lista_lagartos:
            matriz_posicao[l.i, l.j] = l.estrategia

        freq = calcular_freq(matriz_posicao)
        vizinhos_mean = media_vizinhos(lista_lagartos)
        r_por_estrat = media_vizinhos_por_estrategia(lista_lagartos)
        resultados.append({
            "pop": pop,
            "t": -1,
            "freq_O": freq[0],
            "freq_Y": freq[1],
            "freq_B": freq[2],
            "vizinhos_mean": vizinhos_mean,
            "vizinhos_mean_O": r_por_estrat[0],
            "vizinhos_mean_Y": r_por_estrat[1],
            "vizinhos_mean_B": r_por_estrat[2],
        })

        for t in range(n_geracoes):
            print(f"População {pop+1} - Geração {t+1}/{n_geracoes}")

        if sobreposicao == True:
            for _ in range(L*L):  # sorteia n_lagartos vezes, podendo repetir
                i = np.random.randint(0, L)
                j = np.random.randint(0, L)
                lagarto = mapa[(i, j)] # lagarto sorteado
                
                lagarto.calcular_coord_vizinhos(L) # define os vizinhos do lagarto sorteado                
                #lagarto.ajustar_vizinhos_reciprocos(mapa) # ajusta os vizinhos recíprocos do lagarto sorteado
                lagarto.calcular_n_vizinhos()
                lagarto.calcular_fitness(mapa) # calcula o fitness apenas do lagarto sorteado
                
                if lagarto.estrategia == 'O':
                    d = lagarto.mortalidade(A, wO)
                elif lagarto.estrategia == 'Y':
                    d = lagarto.mortalidade(A, wY)
                else:
                    d = lagarto.mortalidade(A, wB)
                
                if np.random.rand() > d: # se o número sorteado for maior que a probabilidade de mortalidade, o lagarto sobrevive e mantém sua estratégia
                    print(f"Lagarto ({i}, {j}) sobreviveu")
                    pass
                else: # se o lagarto morrer, sorteia um vizinho para ocupar a posição do lagarto morto
                    maior_fitness = lagarto.fitness
                    melhor_raio = lagarto.raio
                    melhor_estrategia = lagarto.estrategia
                    
                    for vizinho_coord in lagarto.coord_vizinhos:
                        vizinho = mapa[vizinho_coord]

                        if vizinho.fitness > maior_fitness:
                            maior_fitness = vizinho.fitness
                            melhor_raio = vizinho.raio
                            melhor_estrategia = vizinho.estrategia
                        
                        elif vizinho.fitness == maior_fitness:
                            a = np.random.choice([0, 1])
                            if a > 0.5:
                                melhor_raio = vizinho.raio
                                melhor_estrategia = vizinho.estrategia

                        else:
                            continue

                    lagarto.estrategia = melhor_estrategia # adota uma estratégia aleatória entre as dos vizinhos           
                    lagarto.raio = melhor_raio # adota um raio aleatório entre os dos vizinhos

        else:  # sem sobreposição
            novas_estrategias = {}
            novos_raios = {}
            
            ordem = np.random.permutation(len(lista_lagartos))  # todos os lagartos, 1 vez
            for idx in ordem:
                lagarto = lista_lagartos[idx]
                i, j = lagarto.i, lagarto.j
                lagarto = mapa[(i, j)] # lagarto sorteado

                lagarto.calcular_coord_vizinhos(L)
                lagarto.calcular_n_vizinhos()
                lagarto.calcular_fitness(mapa)  # usa estado atual (geração antiga)

                if lagarto.estrategia == 'O':
                    d = lagarto.mortalidade(A, wO)
                elif lagarto.estrategia == 'Y':
                    d = lagarto.mortalidade(A, wY)
                else:
                    d = lagarto.mortalidade(A, wB)

                if np.random.rand() > d:
                    print(f"Lagarto ({i}, {j}) sobreviveu")
                    pass
                else:
                    maior_fitness = lagarto.fitness
                    melhor_raio = lagarto.raio
                    melhor_estrategia = lagarto.estrategia
                    
                    for vizinho_coord in lagarto.coord_vizinhos:
                        vizinho = mapa[vizinho_coord]

                        if vizinho.fitness > maior_fitness:
                            maior_fitness = vizinho.fitness
                            melhor_raio = vizinho.raio
                            melhor_estrategia = vizinho.estrategia
                        
                        elif vizinho.fitness == maior_fitness:
                            a = np.random.choice([0, 1])
                            if a > 0.5:
                                melhor_raio = vizinho.raio
                                melhor_estrategia = vizinho.estrategia
                        
                        else:
                            continue

                    # guarda a atualização para aplicar depois
                    novas_estrategias[(i, j)] = melhor_estrategia
                    novos_raios[(i, j)] = melhor_raio

            for (i, j), nova_estrategia in novas_estrategias.items():
                mapa[(i, j)].estrategia = nova_estrategia
                mapa[(i, j)].raio = novos_raios[(i, j)]

            # sincroniza matriz depois da aplicação em bloco
            for l in lista_lagartos:
                matriz_posicao[l.i, l.j] = l.estrategia

            t += 1

            freq = calcular_freq(matriz_posicao)
            vizinhos_mean = media_vizinhos(lista_lagartos)
            r_por_estrat = media_vizinhos_por_estrategia(lista_lagartos)

            resultados.append({
                "pop": pop,
                "t": t,
                "freq_O": freq[0],
                "freq_Y": freq[1],
                "freq_B": freq[2],
                "vizinhos_mean": vizinhos_mean,
                "vizinhos_mean_O": r_por_estrat[0],
                "vizinhos_mean_Y": r_por_estrat[1],
                "vizinhos_mean_B": r_por_estrat[2],
            })

    return pd.DataFrame(resultados)


# ==============================
# RODAR
# ==============================

df = simulacao(A = A, wO = wO, wY = wY, wB = wB, seed = 1)
df.to_csv(os.path.join(output_dir, "resultados.csv"), index=False)

print("Simulação concluída.")

População 1 - Geração 1/100
População 1 - Geração 2/100
População 1 - Geração 3/100
População 1 - Geração 4/100
População 1 - Geração 5/100
População 1 - Geração 6/100
População 1 - Geração 7/100
População 1 - Geração 8/100
População 1 - Geração 9/100
População 1 - Geração 10/100
População 1 - Geração 11/100
População 1 - Geração 12/100
População 1 - Geração 13/100
População 1 - Geração 14/100
População 1 - Geração 15/100
População 1 - Geração 16/100
População 1 - Geração 17/100
População 1 - Geração 18/100
População 1 - Geração 19/100
População 1 - Geração 20/100
População 1 - Geração 21/100
População 1 - Geração 22/100
População 1 - Geração 23/100
População 1 - Geração 24/100
População 1 - Geração 25/100
População 1 - Geração 26/100
População 1 - Geração 27/100
População 1 - Geração 28/100
População 1 - Geração 29/100
População 1 - Geração 30/100
População 1 - Geração 31/100
População 1 - Geração 32/100
População 1 - Geração 33/100
População 1 - Geração 34/100
População 1 - Geração 3

In [31]:
# ==============================
# SIMULAÇÃO
# ==============================

def simulacao_gif(A, wO, wY, wB, seed=None):

    resultados = []
    matrizes_posicao_hist = [] # lista para armazenar as matrizes de posição de cada geração
    matrizes_fitness_hist = []

    if seed is not None:
        np.random.seed(seed)

    lista_lagartos = criar_lagartos()
    mapa = {(l.i, l.j): l for l in lista_lagartos}

    matriz_posicao = np.empty((L, L), dtype=object)
    for l in lista_lagartos:
        matriz_posicao[l.i, l.j] = l.estrategia
    
    matrizes_posicao_hist.append(matriz_posicao.copy()) # armazena a matriz de posição da geração inicial

    freq = calcular_freq(matriz_posicao)
    vizinhos_mean = media_vizinhos(lista_lagartos)
    resultados.append({
        "t": -1,
        "freq_O": freq[0],
        "freq_Y": freq[1],
        "freq_B": freq[2],
        "vizinhos_mean": vizinhos_mean
    })

    for t in range(n_geracoes):
        
        if sobreposicao == True:
            for _ in range(L*L):  # sorteia n_lagartos vezes, podendo repetir
                i = np.random.randint(0, L)
                j = np.random.randint(0, L)
                lagarto = mapa[(i, j)] # lagarto sorteado
                
                lagarto.calcular_coord_vizinhos(L) # define os vizinhos do lagarto sorteado                
                #lagarto.ajustar_vizinhos_reciprocos(mapa) # ajusta os vizinhos recíprocos do lagarto sorteado
                lagarto.calcular_n_vizinhos()
                lagarto.calcular_fitness(mapa) # calcula o fitness apenas do lagarto sorteado
                
                if lagarto.estrategia == 'O':
                    d = lagarto.mortalidade(A, wO)
                elif lagarto.estrategia == 'Y':
                    d = lagarto.mortalidade(A, wY)
                else:
                    d = lagarto.mortalidade(A, wB)
                
                if np.random.rand() > d: # se o número sorteado for maior que a probabilidade de mortalidade, o lagarto sobrevive e mantém sua estratégia
                    print(f"Lagarto ({i}, {j}) sobreviveu")
                    pass
                else: # se o lagarto morrer, sorteia um vizinho para ocupar a posição do lagarto morto
                    maior_fitness = lagarto.fitness
                    melhor_raio = lagarto.raio
                    melhor_estrategia = lagarto.estrategia
                    
                    for vizinho_coord in lagarto.coord_vizinhos:
                        vizinho = mapa[vizinho_coord]

                        if vizinho.fitness > maior_fitness:
                            maior_fitness = vizinho.fitness
                            melhor_raio = vizinho.raio
                            melhor_estrategia = vizinho.estrategia
                        
                        elif vizinho.fitness == maior_fitness:
                            a = np.random.choice([0, 1])
                            if a > 0.5:
                                melhor_raio = vizinho.raio
                                melhor_estrategia = vizinho.estrategia

                        else:
                            continue

                    lagarto.estrategia = melhor_estrategia # adota uma estratégia aleatória entre as dos vizinhos           
                    lagarto.raio = melhor_raio # adota um raio aleatório entre os dos vizinhos

        else:  # sem sobreposição
            novas_estrategias = {}
            novos_raios = {}
            ordem = np.random.permutation(len(lista_lagartos))  # todos os lagartos, 1 vez
            for idx in ordem:
                lagarto = lista_lagartos[idx]
                i, j = lagarto.i, lagarto.j
                lagarto = mapa[(i, j)] # lagarto sorteado

                lagarto.calcular_coord_vizinhos(L)
                lagarto.calcular_n_vizinhos()
                lagarto.calcular_fitness(mapa)  # usa estado atual (geração antiga)

                if lagarto.estrategia == 'O':
                    d = lagarto.mortalidade(A, wO)
                elif lagarto.estrategia == 'Y':
                    d = lagarto.mortalidade(A, wY)
                else:
                    d = lagarto.mortalidade(A, wB)

                if np.random.rand() > d:
                    print(f"Lagarto ({i}, {j}) sobreviveu")
                    pass
                else:
                    maior_fitness = lagarto.fitness
                    melhor_raio = lagarto.raio
                    melhor_estrategia = lagarto.estrategia
                    
                    for vizinho_coord in lagarto.coord_vizinhos:
                        vizinho = mapa[vizinho_coord]

                        if vizinho.fitness > maior_fitness:
                            maior_fitness = vizinho.fitness
                            melhor_raio = vizinho.raio
                            melhor_estrategia = vizinho.estrategia
                        
                        elif vizinho.fitness == maior_fitness:
                            a = np.random.choice([0, 1])
                            if a > 0.5:
                                melhor_raio = vizinho.raio
                                melhor_estrategia = vizinho.estrategia
                        
                        else:
                            continue

                    # guarda a atualização para aplicar depois
                    novas_estrategias[(i, j)] = melhor_estrategia
                    novos_raios[(i, j)] = melhor_raio

            for (i, j), nova_estrategia in novas_estrategias.items():
                mapa[(i, j)].estrategia = nova_estrategia
                mapa[(i, j)].raio = novos_raios[(i, j)]

        for l in lista_lagartos:
            matriz_posicao[l.i, l.j] = l.estrategia

        t += 1

        matrizes_posicao_hist.append(matriz_posicao.copy()) # armazena a matriz de posição da geração atual

        freq = calcular_freq(matriz_posicao)
        vizinhos_mean = media_vizinhos(lista_lagartos)

        resultados.append({
            "t": t,
            "freq_O": freq[0],
            "freq_Y": freq[1],
            "freq_B": freq[2],
            "vizinhos_mean": vizinhos_mean
        })

    return pd.DataFrame(resultados), matrizes_posicao_hist


# ==============================
# RODAR
# ==============================

df, matrizes_posicao_hist = simulacao_gif(A=A, wO=wO, wY=wY, wB=wB, seed=1)
#df.to_csv(os.path.join(output_dir, "resultados_gif.csv"), index=False)

print("Simulação concluída.")

Simulação concluída.


In [34]:
import matplotlib.colors as mcolors

cores_grid = {"O": "#FD9800", "B": "#0047B3", "Y": "#FFF237"}

def matriz_para_rgb(matriz):
    # Converte hex para RGB normalizado (0-1)
    return np.array([[mcolors.to_rgb(cores_grid.get(cell, "#FFFFFF")) for cell in row] for row in matriz])

geracao = 50  # escolha a geração que você quer
rgb = matriz_para_rgb(matrizes_posicao_hist[geracao])

plt.figure(figsize=(6, 6))
plt.imshow(rgb)
plt.title(f"8 neighbors, t = {geracao}")
plt.axis("off")
plt.savefig(os.path.join(output_dir, f"geracao_{geracao}.png"),
            dpi=300, bbox_inches="tight")
plt.close()

In [33]:
# gerando o GIF das posições

cores_grid = {"O": "#FD9800", "B": "#0047B3", "Y": "#FFF237"}

def matriz_para_rgb(matriz):
    # Converte hex para RGB normalizado (0-1)
    return np.array([[mcolors.to_rgb(cores_grid.get(cell, "#FFFFFF")) for cell in row] for row in matriz])

# Crie a figura
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    ax.clear()
    ax.imshow(matriz_para_rgb(matrizes_posicao_hist[frame]))
    ax.set_title(f"Geração {frame}")
    ax.axis('off')

ani = animation.FuncAnimation(
    fig, update, frames=len(matrizes_posicao_hist), interval=100, repeat=False
)

# Salvar como GIF
ani.save(os.path.join(output_dir, "simulacao_grid.gif"), writer='pillow', fps=10)
ani.save(os.path.join(output_dir, "simulacao_grid.mp4"), writer='ffmpeg', fps=10)
plt.close()